# Clustering Experiments — User Request Pattern Discovery

Exploratory analysis of 1,668 user turns from `user_nl_root_only` corpus view.

**Strategy**: MiniLM-L6 sentence embeddings → UMAP dimensionality reduction → HDBSCAN density clustering → LLM labelling via `gpt-5.4-mini`.

**Goal**: find semantically coherent request groups that can guide episode-level skill mining.

**Key result**: silhouette jumps from 0.23 (KMeans) to **0.77** (HDBSCAN+UMAP), yielding **28 interpretable clusters** with LLM-assigned skill names.

## 1. Setup

In [ ]:
from pathlib import Path
import json, numpy as np, pickle
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import TfidfVectorizer
import umap, hdbscan
import warnings; warnings.filterwarnings('ignore')

REPO   = Path('.')
VIEW   = REPO / 'artifacts/chat-analysis/views/user_nl_root_only/corpus_view.jsonl'
EP_DIR = REPO / 'artifacts/chat-analysis/episodes/smoke/default'
import os
BASE_URL = os.environ.get('SKILLDRILLA_LLM_BASE_URL', 'https://api.openai.com/v1')
API_KEY  = os.environ.get('SKILLDRILLA_LLM_API_KEY', '')
MODEL    = 'gpt-5.4-mini'

print('Setup complete')

## 2. Load corpus view

`user_nl_root_only` = every user natural-language turn from root sessions (no subagent, no tool, no system records).

In [ ]:
rows = []
for line in VIEW.read_text().splitlines():
    if line.strip():
        rows.append(json.loads(line))

texts_raw    = [r['evidence'].get('content_text','') or '' for r in rows]
projects     = [r['evidence'].get('project_slug','') for r in rows]
evidence_ids = [r['evidence'].get('evidence_id','') for r in rows]

print(f'Total rows:      {len(rows)}')
print(f'Unique projects: {len(set(projects))}')
print('\nTop 10 projects:')
for slug, cnt in Counter(projects).most_common(10):
    print(f'  {cnt:4d}  {slug or "(blank)"}')

## 3. Noise filtering

Remove IDE context injections, interrupted requests, health-check probes, and trivially short strings.

In [ ]:
NOISE_EXACT  = {'[request interrupted by user]', '[request interrupted by user for tool use]',
                'respond with exactly "hello!" and nothing else.', 'test', 'adwa', 'dada'}
NOISE_PREFIX = ('<ide_opened_file>', '<ide_selection>')

keep = [i for i, t in enumerate(texts_raw)
        if t.lower().strip() not in NOISE_EXACT
        and not any(t.lower().startswith(p.lower()) for p in NOISE_PREFIX)
        and len(t.strip()) >= 20]

clean_texts  = [texts_raw[i] for i in keep]
clean_rows   = [rows[i] for i in keep]
clean_ev_ids = [evidence_ids[i] for i in keep]
print(f'Kept {len(clean_texts)} / {len(texts_raw)} rows ({len(texts_raw)-len(clean_texts)} removed)')

## 4. Sentence embeddings — MiniLM-L6-v2

384-dimensional, L2-normalised. Locally cached (~23 MB model). Cosine similarity = dot product on normalised vectors.

In [ ]:
model_st = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model_st.encode(clean_texts, batch_size=64,
                             show_progress_bar=True, normalize_embeddings=True)
print(f'Embeddings: {embeddings.shape}')

## 5. Baseline: KMeans & Ward-linkage agglomerative

Establish how well convex-cluster methods perform on this embedding space.

In [ ]:
print(f"{'k':>6}  {'KMeans':>10}  {'Ward':>10}")
print('-'*30)
for k in [10, 15, 20, 25, 30, 40, 50]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(embeddings)
    ks = silhouette_score(embeddings, km, metric='cosine', sample_size=500)
    wd = AgglomerativeClustering(n_clusters=k, linkage='ward').fit_predict(embeddings)
    ws = silhouette_score(embeddings, wd, metric='cosine', sample_size=500)
    print(f'{k:>6}  {ks:>10.4f}  {ws:>10.4f}')

> **Result**: best silhouette ~0.23. Text embeddings form non-convex manifolds — KMeans/Ward can't capture the real structure.

## 6. UMAP dimensionality reduction

Project 384D → 10D preserving local neighbourhood structure. HDBSCAN operates on this reduced space.

In [ ]:
reducer = umap.UMAP(n_components=10, n_neighbors=15, min_dist=0.0,
                    metric='cosine', random_state=42)
emb_10d = reducer.fit_transform(embeddings)
print(f'UMAP 10D: {emb_10d.shape}')

## 7. HDBSCAN parameter sweep

Density-based clustering with no pre-set k. Low-density points become noise (label −1).

In [ ]:
print(f"{'min_cs':>8}  {'clusters':>8}  {'noise%':>7}  {'silhouette':>10}")
print('-'*40)
for mcs in [8, 12, 15, 20, 25, 30]:
    h = hdbscan.HDBSCAN(min_cluster_size=mcs, metric='euclidean',
                         cluster_selection_method='eom')
    lbl = h.fit_predict(emb_10d)
    nc  = len(set(lbl)) - (1 if -1 in lbl else 0)
    pct = (lbl==-1).mean()*100
    mask = lbl != -1
    sil  = silhouette_score(emb_10d[mask], lbl[mask]) if mask.sum()>1 else float('nan')
    print(f'{mcs:>8}  {nc:>8}  {pct:>6.1f}%  {sil:>10.4f}')

> **Silhouette jumps from 0.23 → 0.77.** Density-based clustering on UMAP space captures the real structure.
>
> **Trade-off**: min_cs=8 → 51 fine clusters; min_cs=15 → 28 cleaner clusters. We proceed with **min_cs=15** for interpretability.

## 8. Cluster inspection — 28 clusters (min_cs=15)

Each cluster with its size, top TF-IDF discriminative terms, and representative samples.

In [ ]:
hdb = hdbscan.HDBSCAN(min_cluster_size=15, metric='euclidean',
                       cluster_selection_method='eom')
labels = hdb.fit_predict(emb_10d)
counts = Counter(labels)
real   = sorted((l for l in counts if l!=-1), key=lambda l: -counts[l])

vec  = TfidfVectorizer(max_features=600, stop_words='english', ngram_range=(1,2))
vec.fit(clean_texts)
feat = vec.get_feature_names_out()

print(f'Clusters: {len(real)}  |  Noise: {counts.get(-1,0)} ({counts.get(-1,0)/len(labels)*100:.1f}%)')
print()
for lbl in real:
    idxs = [i for i,l in enumerate(labels) if l==lbl]
    X    = vec.transform([clean_texts[i] for i in idxs])
    top  = [feat[j] for j in np.asarray(X.mean(0)).flatten().argsort()[-5:][::-1]]
    rep  = clean_texts[idxs[0]][:90].replace('\n',' ')
    print(f'[{lbl:2d}] {counts[lbl]:4d}  {"  ".join(top[:4])}')
    print(f'      {rep}…')
    print()

## 9. LLM cluster labelling — gpt-5.4-mini

Feed each cluster's TF-IDF terms and sample texts to `gpt-5.4-mini` via the local OpenAI-compatible endpoint.
Ask for a `skill_name` (snake_case), a one-line description, and an `is_noise` flag.

In [ ]:
from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

class ClusterLabel(BaseModel):
    cluster_id: int
    skill_name: str
    one_line: str
    is_noise: bool

class LabelResult(BaseModel):
    labels: list[ClusterLabel]

provider = OpenAIProvider(base_url=BASE_URL, api_key=API_KEY)
llm      = OpenAIChatModel(MODEL, provider=provider)
agent    = Agent(llm, output_type=LabelResult)

# Build cluster summaries for prompt
cluster_blobs = []
for lbl in real:
    idxs    = [i for i,l in enumerate(labels) if l==lbl]
    ctexts  = [clean_texts[i] for i in idxs]
    X       = vec.transform(ctexts)
    top     = [feat[j] for j in np.asarray(X.mean(0)).flatten().argsort()[-6:][::-1]]
    samps   = [t[:120].replace('\n',' ') for t in ctexts[:4]]
    cluster_blobs.append({'id': int(lbl), 'size': counts[lbl], 'top_terms': top, 'samples': samps})

cluster_text = '\n\n'.join(
    f"Cluster {c['id']} (n={c['size']}):\n"
    f"  terms: {', '.join(c['top_terms'])}\n"
    f"  samples:\n" + '\n'.join(f'    - {s}' for s in c['samples'])
    for c in cluster_blobs)

prompt = (
    f'You are analysing clusters of user messages sent to an AI coding assistant.\n'
    f'Label each cluster with:\n'
    f'- skill_name: short snake_case identifier for what the user is trying to DO\n'
    f'- one_line: ≤10 words describing the user activity\n'
    f'- is_noise: true ONLY if system-injected boilerplate, not real user intent\n\n'
    f'Clusters:\n{cluster_text}')

result = agent.run_sync(prompt)
named  = {lbl.cluster_id: lbl for lbl in result.output.labels}

print(f"{'id':>4}  {'n':>5}  {'noise':>5}  skill_name")
print('-'*70)
for lbl in real:
    n = named.get(lbl)
    if n:
        tag = 'NOISE' if n.is_noise else '     '
        print(f'{lbl:>4}  {counts[lbl]:>5}  {tag}  {n.skill_name}')

### Cluster label table

| Cluster | Size | Skill | Noise? |
|---|---|---|---|
| 0 | 132 | `boilerplate_prompt_injection` | yes |
| 26 | 118 | `continue_or_check_task_status` | no |
| 23 | 104 | `doc_hub_search_configuration` | no |
| 24 | 69 | `pydantic_ai_docs_integration` | no |
| 22 | 65 | `job_timeout_and_persistence_fix` | no |
| 20 | 63 | `pipeline_config_and_script_update` | no |
| 9 | 63 | `playwright_native_testing_plan` | no |
| 14 | 46 | `milestone_verification_planning` | no |
| 2 | 45 | `test_driven_bugfixing` | no |
| 21 | 37 | `install_cli_proxy_api` | no |
| 10 | 37 | `claude_skill_and_subagent_management` | no |
| 1 | 34 | `skill_specification_design` | no |
| 4 | 33 | `milestone_plan_decomposition` | no |
| 3 | 32 | `plan_status_inspection` | no |
| 19 | 32 | `adversarial_plan_review` | no |
| 17 | 31 | `agent_phase_role_mapping` | no |
| 15 | 29 | `workflow_control_button_creation` | no |
| 12 | 29 | `adr_pipeline_phase_management` | no |
| 18 | 27 | `remediation_and_completion_reporting` | no |
| 5 | 25 | `prompt_output_and_response_tuning` | no |
| 7 | 22 | `browser_platform_architecture_refinement` | no |
| 8 | 22 | `claude_md_skill_creation` | no |
| 16 | 21 | `notion_task_instructions_following` | no |
| 27 | 20 | `network_issue_diagnosis` | no |
| 11 | 18 | `model_selection_and_fallback_setup` | no |
| 13 | 18 | `milestone_plan_refinement` | no |
| 25 | 17 | `architecture_document_refinement` | no |
| 6 | 17 | `claude_llm_app_or_rlm_setup` | no |


## 10. 2D UMAP visualisation

In [ ]:
reducer2d = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.05,
                      metric='cosine', random_state=42)
emb2d = reducer2d.fit_transform(embeddings)

import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 10))
cmap = plt.cm.get_cmap('tab20', len(real))
for idx, lbl in enumerate(real):
    mask = labels == lbl
    nm   = named.get(lbl)
    name = nm.skill_name if hasattr(nm, 'skill_name') else str(lbl)
    ax.scatter(emb2d[mask,0], emb2d[mask,1], s=20, alpha=0.7,
              label=f'{lbl}: {name[:30]}', color=cmap(idx))

ax.scatter(emb2d[labels==-1,0], emb2d[labels==-1,1],
           s=8, alpha=0.15, color='grey', label='noise')
ax.legend(bbox_to_anchor=(1.02,1), loc='upper left', fontsize=6, ncol=1)
ax.set_title('UMAP 2D — user turn clusters (HDBSCAN min_cs=15)')
plt.tight_layout()
plt.savefig('artifacts/chat-analysis/semantic/skill-mining/umap_clusters.png', dpi=150)
plt.show()
print('Saved umap_clusters.png')

## 11. Deep dive — top actionable clusters

### Cluster 26: `continue_or_check_task_status` (n=118)

The single most common non-boilerplate pattern. User checks progress, confirms plans, asks for git status. Strong candidate for a quick-status skill.

In [ ]:
c26 = [clean_texts[i] for i,l in enumerate(labels) if l==26]
c26_projs = Counter(clean_rows[i]['evidence']['project_slug']
                       for i,l in enumerate(labels) if l==26)
print(f'Size: {len(c26)}')
print(f'Projects: {c26_projs.most_common(5)}')
print()
for t in c26[:4]:
    print(f'  · {t[:130].replace(chr(10)," ")}')
    print()

### Cluster 23: `doc_hub_search_configuration` (n=104)

Repeated work configuring and invoking the doc-hub search skill/subagent across sessions and projects.

In [ ]:
c23 = [clean_texts[i] for i,l in enumerate(labels) if l==23]
c23_projs = Counter(clean_rows[i]['evidence']['project_slug']
                       for i,l in enumerate(labels) if l==23)
print(f'Size: {len(c23)}')
print(f'Projects: {c23_projs.most_common(5)}')
print()
for t in c23[:4]:
    print(f'  · {t[:130].replace(chr(10)," ")}')
    print()

### Cluster 4: `milestone_plan_decomposition` (n=33)

User consistently gives a plan file path and asks the model to implement the next incomplete milestone. Highly formulaic — top candidate for `run_next_milestone`.

In [ ]:
c4 = [clean_texts[i] for i,l in enumerate(labels) if l==4]
c4_projs = Counter(clean_rows[i]['evidence']['project_slug']
                       for i,l in enumerate(labels) if l==4)
print(f'Size: {len(c4)}')
print(f'Projects: {c4_projs.most_common(5)}')
print()
for t in c4[:4]:
    print(f'  · {t[:130].replace(chr(10)," ")}')
    print()

### Cluster 19: `adversarial_plan_review` (n=32)

Recurring adversarial review loop: plan → adversarial critique → refinement. Used across doc-hub, pydantic-ai-docs, stagehand, and playwright projects.

In [ ]:
c19 = [clean_texts[i] for i,l in enumerate(labels) if l==19]
c19_projs = Counter(clean_rows[i]['evidence']['project_slug']
                       for i,l in enumerate(labels) if l==19)
print(f'Size: {len(c19)}')
print(f'Projects: {c19_projs.most_common(5)}')
print()
for t in c19[:4]:
    print(f'  · {t[:130].replace(chr(10)," ")}')
    print()

### Cluster 2: `test_driven_bugfixing` (n=45)

Run tests, fix failures, iterate. Short TDD feedback loops.

In [ ]:
c2 = [clean_texts[i] for i,l in enumerate(labels) if l==2]
c2_projs = Counter(clean_rows[i]['evidence']['project_slug']
                       for i,l in enumerate(labels) if l==2)
print(f'Size: {len(c2)}')
print(f'Projects: {c2_projs.most_common(5)}')
print()
for t in c2[:4]:
    print(f'  · {t[:130].replace(chr(10)," ")}')
    print()

### Cluster 8: `claude_md_skill_creation` (n=22)

Create CLAUDE.md files, define new skills, bootstrap skill directories.

In [ ]:
c8 = [clean_texts[i] for i,l in enumerate(labels) if l==8]
c8_projs = Counter(clean_rows[i]['evidence']['project_slug']
                       for i,l in enumerate(labels) if l==8)
print(f'Size: {len(c8)}')
print(f'Projects: {c8_projs.most_common(5)}')
print()
for t in c8[:4]:
    print(f'  · {t[:130].replace(chr(10)," ")}')
    print()

## 12. LLM skill mining on top clusters

Feed each cluster's turns to `gpt-5.4-mini` and ask it to define the pattern as a reusable **skill specification**.

This is fundamentally different from random sampling — the LLM sees turns already known to be similar, so it only needs to **name and describe** the pattern, not discover it from noise.

In [ ]:
class SkillSpec(BaseModel):
    skill_name: str
    trigger_description: str
    input_contract: str
    output_contract: str
    example_invocations: list[str]
    confidence: float

spec_agent = Agent(llm, output_type=SkillSpec)
TARGET = [4, 19, 2, 8]

specs = {}
for cid in TARGET:
    turns = [clean_texts[i] for i,l in enumerate(labels) if l==cid]
    excerpts = '\n'.join(f'[{j+1}] {t[:180]}' for j,t in enumerate(turns[:25]))
    prompt = (
        f'Below are {min(len(turns),25)} user messages from the same semantic cluster.\n'
        f'They represent a recurring request pattern sent to an AI coding assistant.\n\n'
        f'Messages:\n{excerpts}\n\n'
        f'Define this as a reusable skill.')
    spec = spec_agent.run_sync(prompt).output
    specs[cid] = spec
    print(f'Cluster {cid}: {spec.skill_name} (conf={spec.confidence:.2f})')
    print(f'  Trigger: {spec.trigger_description[:100]}')
    print(f'  Input:   {spec.input_contract[:100]}')
    print(f'  Output:  {spec.output_contract[:100]}')
    print()

## 13. Extracted skill specifications

### `run_next_milestone` (cluster 4, confidence 0.97)

**Trigger**: User provides a plan file path and asks to implement the next incomplete milestone

**Input**: Plan file path (markdown); optional: branch name, output directory

**Output**: Milestone implementation, status update written back to plan file, commit

**Example invocations**:
- _plan file: packages/pydantic-ai-docs/plans/overview.md Implement the next milestone_
- _Read the plan and determine if there are milestones not yet marked complete_

### `adversarial_plan_review` (cluster 19, confidence 0.93)

**Trigger**: User asks for adversarial critique of an implementation plan, often specifying round number

**Input**: Plan text (markdown); optional: round number, reviewer persona

**Output**: Structured adversarial review with weaknesses, risks, and recommended mitigations

**Example invocations**:
- _You are a senior adversarial reviewer performing refinement round 3/4_
- _plan folder: packages/pydantic-ai-docs/plans — adversarial review and refine_

### `test_and_fix_loop` (cluster 2, confidence 0.88)

**Trigger**: User asks to run tests, inspect failures, and fix iteratively until passing

**Input**: Test command or test file path; optional: target test name

**Output**: Fixed code with passing tests; summary of what changed

**Example invocations**:
- _Run the test suite for RLM-Claude-Code_
- _Recommend a small set of ~5 parameter combinations to test_

### `bootstrap_claude_skill` (cluster 8, confidence 0.84)

**Trigger**: User asks to create a new Claude Code skill, CLAUDE.md, or skill directory

**Input**: Skill name and description; optional: base directory path

**Output**: Skill directory with CLAUDE.md, README, and initial implementation scaffold

**Example invocations**:
- _Please analyze this codebase and create a CLAUDE.md file_
- _Base directory for this skill: ~/.claude/skills/adversarial-pipeline_

## 14. Summary and next steps

### Method comparison

| Approach | Silhouette | Clusters | Notes |
|---|---|---|---|
| KMeans (raw 384D) | 0.23 | 50 | Poor — convex assumption fails |
| Agglomerative Ward (raw) | 0.20 | 30 | Same limitation |
| **HDBSCAN + UMAP 10D** | **0.77** | **28** | Captures non-convex manifold |

### Three-layer pattern taxonomy

1. **Infrastructure/tooling** (clusters 21, 27, 37, 8): installing proxies, network debugging, creating skills, managing git
2. **Pipeline orchestration** (clusters 4, 19, 17, 12, 14, 18): adversarial review loops, milestone execution, ADR phases
3. **Domain-specific development** (clusters 23, 24, 9, 7, 15): doc-hub, pydantic-ai, playwright/stagehand, camoufox, LightMock

### Top skill candidates (confidence × recurrence)

| Rank | Skill | Cluster | Size | Confidence |
|---|---|---|---|---|
| 1 | `run_next_milestone` | 4 | 33 | 0.97 |
| 2 | `adversarial_plan_review` | 19 | 32 | 0.93 |
| 3 | `test_and_fix_loop` | 2 | 45 | 0.88 |
| 4 | `bootstrap_claude_skill` | 8 | 22 | 0.84 |
| 5 | `doc_hub_search_skill` | 23 | 104 | 0.81 |
| 6 | `task_status_check` | 26 | 118 | 0.78 |

### Next steps

1. **Episode-level join**: link clusters back to episodes to find which skills co-occur in a single conversation
2. **Cluster-guided skill mining**: pass cluster representatives (instead of random samples) as episode-mining input
3. **Skill validation**: trace representative `evidence_id` values back to raw transcripts and manually verify boundaries
4. **Candidate → real skill**: for the top-4 skills, generate actual Claude Code skill scaffolds and validate end-to-end


---

# Part 2: Episode-Level Skill Mining

Single-turn clustering (Part 1) found **28 semantic clusters** of similar user messages. But it can't tell us what the user is *actually accomplishing* across a multi-turn conversation.

This section feeds full **episode transcripts** (user + assistant natural language only, no tool calls) to `gpt-5.4-mini` to understand workflow arcs, extract implicit playbooks, and discover skill chains.

**Method**: Cluster-guided episode selection → Arc shape classification → Deep playbook extraction → Skill chain discovery → Synthesis and ranking.

**LLM budget**: 14 calls to gpt-5.4-mini (of 20 budgeted). Each under ~4000 tokens.


## 15. Phase 1: Arc Shape Classification

18 episodes selected via cluster-guided scoring (prioritising multi-cluster episodes with moderate length). Each episode's user+assistant NL transcript was compressed and sent to gpt-5.4-mini for arc classification.

| Arc Shape | Count | Description |
|---|---|---|
| `multi_phase` | 12 | 2+ distinct phases (e.g. plan → implement → review → fix) |
| `plan_then_execute` | 3 | Agree on plan first, then systematically execute it |
| `iterative_refine` | 1 | Request → review → adjust, converging on result |
| `exploratory` | 1 | Investigation/learning, no fixed end goal |
| `debug_loop` | 1 | Failure → diagnose → fix → test cycles |


> **Key finding**: 12 of 18 episodes are `multi_phase` — the developer's typical workflow is not a single request-response cycle. It's a structured multi-phase process: plan → review → implement → test → fix. This is exactly what single-turn clustering misses.

### Sample episode arcs

- `ec8c0b0fd480e4d6…` — **iterative_refine** — Document what the assembly folder does by tracing its flow into a Mermaid diagra  
  Phases: _inspect folder, map workflow to Mermaid markdown, clarify agent roles from config_
- `55f5dedfa11a04cd…` — **multi_phase** — Build a very simple end-to-end Python Agent SDK example that sends prompts throu  
  Phases: _brainstorm solution shape, choose language/runtime, write minimal Agent SDK script_
- `2724ae0adce4027a…` — **multi_phase** — Understand and document the doc-hub package architecture, then extend it to hand  
  Phases: _explore package structure, write README and architecture evaluation, analyze doc-change scenarios_
- `86d31853e38b165a…` — **multi_phase** — Re-implement the iterative deepthink-style refinement workflow from the referenc  
  Phases: _study reference repository, reconstruct deepthink workflow, create project scaffolding and docs_
- `2027696024e792f3…` — **multi_phase** — Create an adversarial analysis pipeline for an implementation plan, make it runn  
  Phases: _verify skill availability, ingest adversarial pipeline spec, generate pipeline script_
- `73b69feccfc115a9…` — **multi_phase** — Locate the repo’s developer docs, prioritize agentic-engineering setup needs, re  
  Phases: _repo exploration, requirements prioritization, requirements documentation_


## 16. Phase 2: Deep Playbook Extraction

The 6 highest-scoring episodes received full playbook extraction — asking gpt-5.4-mini to identify the implicit sequence of actions, decision points, delegation patterns, and automation opportunities.

### Episode `2724ae0adce4027a…` — multi_phase
_Understand and document the doc-hub package architecture, then extend it to handle remote documentat_

**Playbook steps** (top 6):
  1. Survey the codebase and read the relevant package files in parallel.
  2. Verify implementation details by checking key modules, line counts, and completeness.
  3. Summarize the architecture and identify gaps or missing behavior.
  4. Cross-check related packages or prior implementations to understand existing patterns.
  5. Translate the gap analysis into a concrete fix plan, including tests and documentation updates.
  6. Implement the code changes iteratively across the affected modules.

**Key decision points**:
  - Decide whether the current package already satisfies the documented behavior or needs a fix, based o
  - Decide how to represent remote-document changes by comparing fetch strategy, manifest handling, and 
  - Decide whether to patch an existing database row versus recreating or migrating configuration, using

**Automation potential**:
  - Automate repository scanning, file inventory, and line-count/completeness checks.
  - Automate detection of missing configuration, schema mismatches, and manifest/caching inconsistencies
  - Automate generation of initial fix plans and test scaffolding from identified gaps.

**Friction**:
  - The user interrupts and redirects mid-stream, indicating shifting requirements or evolving understan
  - JSON escaping in SQL/config updates is called out as a practical editing hazard.


### Episode `5c8590bb4893d82a…` — multi_phase
_Design a more convenient way to add corpora to doc-hub, refine the command-line and logging UX, writ_

**Playbook steps** (top 6):
  1. Identify the current workflow and constraints for adding a corpus.
  2. Propose a more convenient user-facing command flow.
  3. Choose an explicit strategy-based interface instead of URL auto-detection.
  4. Define strategy-specific required flags and validation rules.
  5. Decide how fetcher success/failure should be surfaced in the first version.
  6. Place the new `add` and `logs` commands under a `utilities` subcommand.

**Key decision points**:
  - Whether to keep corpus onboarding as a low-level MCP/JSON flow or add a higher-level CLI.
  - Whether to infer the strategy from the URL or require the user to specify it explicitly.
  - How much validation to encode in the strategy layer versus leaving to implementation.

**Automation potential**:
  - Auto-generate CLI help, manpages, and reference docs from command definitions.
  - Automate tests for strategy-required flags and command wiring.
  - Automate smoke tests after implementation to catch regressions.

**Friction**:
  - Initial corpus addition required either an MCP tool with JSON args or a direct CLI path, which felt 
  - The user rejected auto-detecting the strategy from the URL, forcing an explicit design choice.


### Episode `7f53010ebf9e2e47…` — multi_phase
_This episode starts by reconciling competing design documents, then shifts into architecting an adve_

**Playbook steps** (top 6):
  1. Inventory the relevant specs, architecture docs, and roadmap before making changes.
  2. Cross-check the documents for contradictions, stale references, and duplicated concepts.
  3. Revise the canonical documents first so terminology and responsibilities are aligned.
  4. Choose the implementation strategy for the pipeline architecture and phase execution model.
  5. Select the agent framework, model family, and orchestration approach for the build.
  6. Map the roadmap into concrete phases, branches, or modules with clear ownership.

**Key decision points**:
  - Which source documents are authoritative when newer docs conflict with older ones.
  - Whether to fix the documentation first or proceed directly to implementation scaffolding.
  - Which model/provider stack to use for the agent workflow.

**Automation potential**:
  - Automated detection of stale references and terminology mismatches across docs.
  - Batch rewriting of affected sections after a canonical architecture change.
  - Scaffolding of repository structure, branches, and phase modules from the roadmap.

**Friction**:
  - Repeated document drift between older specs and newer architecture docs.
  - Back-and-forth over which document set should define the implementation baseline.


### Episode `5dd3be789e0fdbfe…` — multi_phase
_Produce a thorough implementation plan, validate third-party API assumptions, then implement and ite_

**Playbook steps** (top 6):
  1. State the deliverable and scope up front, including any skill/base-directory context needed to ground the work.
  2. Request a thorough, architecture-aligned implementation plan rather than jumping straight into code.
  3. Direct the assistant to consult supporting docs or sub-agents when the plan depends on external API details (for example, Pydantic AI usage).
  4. Have the assistant read the relevant architecture/spec documents completely before drafting the plan.
  5. Ask for verification of uncertain API patterns against current documentation before finalizing assumptions.
  6. Run a self-review against the spec and architecture to catch mismatches early.

**Key decision points**:
  - Whether to produce an implementation plan first or begin coding immediately, based on the maturity o
  - Whether the task depends on fresh library/API knowledge, in which case the user delegates to DocSear
  - Whether to trust existing assumptions about an external SDK/API or validate the exact signature and 

**Automation potential**:
  - Reading and comparing architecture/spec documents is highly automatable.
  - API signature verification against current docs can be automated with sub-agents or retrieval.
  - Self-review for coverage gaps and inconsistent assumptions can be automated as a checklist.

**Friction**:
  - Uncertainty about current Pydantic AI API usage required extra verification and background checking.
  - The assistant had to correct an incorrect assumption about the Agent SDK `query()` signature.


### Episode `9ed084a491bfae29…` — multi_phase
_Clarify packaging/install expectations, update user-facing docs, investigate browsing/navigation met_

**Playbook steps** (top 6):
  1. Inspect repository setup to determine how a CLI should be installed and invoked.
  2. Clarify packaging and distribution assumptions before editing docs or code.
  3. Update user-facing documentation to present the correct install paths and repo-based installation options.
  4. Check repository metadata and existing files to infer the current package manager and release strategy.
  5. Expand or revise docs to cover all supported install modes, including direct-from-repo workflows.
  6. Commit documentation changes once the install story is aligned with the actual repo setup.

**Key decision points**:
  - Decide whether the tool should be installed globally, via uv, or via repo-local mechanisms based on 
  - Decide whether the project uses uv as a package manager by checking for lockfiles and environment ma
  - Decide how much installation detail belongs in the README versus historical or internal plan documen

**Automation potential**:
  - Automatically detect package manager conventions from repository markers and infer recommended insta
  - Generate install documentation variants for pipx, uv tool install, and repo-url workflows from a tem
  - Scan codebase structure to identify relevant files, constants, and CLI entrypoints before editing.

**Friction**:
  - Docs did not say whether a global pip install was needed, forcing repository inspection.
  - The repo did not use uv as expected, so package-manager assumptions had to be corrected manually.


### Episode `2027696024e792f3…` — multi_phase
_Create an adversarial analysis pipeline for an implementation plan, make it runnable in the local en_

**Playbook steps** (top 6):
  1. Inspect available skills and confirm whether the desired skill exists.
  2. Load the chosen skill from its base directory and read the associated docs and implementation plan.
  3. Design the pipeline around the source-of-truth documents and define how phases read and compare them.
  4. Implement the pipeline scripts and verify syntax and end-to-end behavior in the local environment.
  5. Address environment prerequisites such as .env values, API connectivity, and offline package availability.
  6. Revise preflight and bootstrap steps to match the real execution environment.

**Key decision points**:
  - Whether the requested skill already exists or needs to be created from scratch.
  - Whether the pipeline should rely on hardcoded page IDs or dynamically read all workspace files.
  - Whether the pipeline should treat every document uniformly or use named agents/waves to group docume

**Automation potential**:
  - Automatically inspect available skills and enumerate loaded capabilities.
  - Automatically scan the workspace for docs and cluster them into processing waves by count or metadat
  - Automatically generate phase prompts from existing specs and implementation plans.

**Friction**:
  - The requested skill did not exist initially, requiring fallback clarification and manual setup.
  - Environment was incomplete: missing `.env` and CLIProxyAPI was not running.


## 17. Phase 3: Skill Chain Discovery

Analysed the cluster-label sequences across all 18 episodes to find recurring workflow chains.

### Recurring chains

- doc_hub → task_status (often the first planning/status handoff)
- skill_design → task_status and task_status → pipeline_script (design turns into execution)
- skill_mgmt → test_bugfix → skill_mgmt (tight fix/adjust loop)
- pydantic_ai → model_select / agent_phase / run_milestone (AI setup or execution branching)
- adversarial_review → claude_md → pipeline_script / task_status (review feeds back into implementation)
- browser_arch → playwright → task_status (browser/UI work validated back into status)
- network_diag → job_timeout → test_bugfix (troubleshoot → fix loop)

### Workflow archetypes

- 1) Planning → implement → verify loop: doc_hub/task_status/pipeline_script/skill_design with frequent adversarial_review and claude_md feedback. Episodes: 7f530..., 84cb8..., 9ed084..., 5c859..., a7e91...
- 2) Fix-forward QA loop: skill_mgmt/test_bugfix/milestone_verify/remediation, often with model_select or rlm_setup. Episodes: fa604..., 6eaa..., a697..., 55f5...
- 3) AI-agent bootstrap / orchestration: pydantic_ai with model_select, agent_phase, run_milestone, remediation. Episodes: 5dd3..., ec8c..., 7930..., 86d318..., 73b69...
- 4) Browser/UI automation workflow: browser_arch and playwright leading back to task_status and reviews. Episodes: e634..., 429721...
- 5) Troubleshooting / reliability workflow: network_diag, job_timeout, adr_pipeline, then test_bugfix or task_status. Episodes: a697..., 5dd3..., 86d318...


### task_status role

> task_status is mostly glue activity and phase delimiter, not a distinct technical phase. It commonly records handoff, progress, or validation after design, implementation, review, or debugging steps. In a few episodes it behaves like a lightweight milestone gate, but its main function is to connect and sequence other workstreams.

### Proposed skills from chain analysis

- **planning-to-implementation handoff**: A work item starts in docs/specs/design and needs to become executable work.
- **review-to-fix loop**: An adversarial review, critique, or markdown review uncovers issues in a workflow or artifact.
- **skill/module repair loop**: A skill or module fails tests, behaves incorrectly, or needs tuning.
- **AI agent bootstrap orchestration**: A new AI workflow, agent, or milestone needs to be set up or advanced.
- **browser UI verification**: A change affects browser behavior, UI flow, or end-to-end interaction.
- **reliability triage and timeout recovery**: Jobs stall, time out, or exhibit unstable execution.
- **docs and architecture consistency audit**: Repository docs, manifests, or architecture notes may be stale or inconsistent.
- **command surface and help generation**: A CLI or tool interface changes, or docs/help output is missing.

## 18. Phase 4: Final Skill Ranking

All playbook extractions, chain analysis, and candidate skills were merged, deduplicated, and scored.

| Rank | Skill | Freq | Auto | Time | Compound | Total |
|---|---|---|---|---|---|---|
| 1 | **docs and architecture consistency audit** | 4 | 5 | 4 | 5 | **18** |
| 2 | **planning-to-implementation handoff** | 5 | 3 | 3 | 4 | **15** |
| 3 | **command surface and help generation** | 2 | 5 | 4 | 4 | **15** |
| 4 | **AI agent bootstrap orchestration** | 2 | 4 | 4 | 5 | **15** |


### Recommendation

Prioritize the docs and architecture consistency audit first, because it is both highly automatable and broadly protective against implementation drift. Next, invest in planning-to-implementation handoff to reduce rework on a frequent, upstream activity. After that, choose between command surface and help generation or AI agent bootstrap orchestration based on current roadmap: pick command surface work if you need immediate productivity gains in CLI projects, or AI agent bootstrap if you are actively standing up new workflow infrastructure with high leverage.

### Identified gaps

- No proposed skill captures the iterative_refine/debug-loop pattern: run something, observe failure or instability, patch, and re-run.
- The catalog does not explicitly cover reliability triage or timeout recovery, even though the arc distribution shows a distinct non-plan-heavy recovery mode.
- Browser/UI verification is also absent; none of the episodes centered on end-to-end frontend interaction checks.

## 19. Combined Analysis Summary

### What single-turn clustering found (Part 1)
- 28 semantic clusters from UMAP+HDBSCAN (silhouette 0.77)
- Keyword-level groupings: what individual messages are *about*
- Top patterns: task status checks, doc-hub work, milestone execution

### What episode-level mining added (Part 2)
- 12/18 selected episodes are **multi_phase** workflows, not simple request-response
- The developer follows a consistent **plan → review → implement → test** meta-workflow
- `task_status` is workflow glue, not a distinct skill — it appears between phases as a checkpoint
- Real skill chains: `doc_hub → task_status` (planning handoff), `skill_mgmt → test_bugfix → skill_mgmt` (fix-forward loop), `adversarial_review → claude_md → pipeline_script` (review → implement)

### Single-turn vs multi-turn: what changed

| Insight | Single-turn | Multi-turn |
|---|---|---|
| What user asks about | Yes (clusters) | — |
| What user is trying to accomplish | No | Yes (arc shape + purpose) |
| What sequence of steps they follow | No | Yes (playbook extraction) |
| What could be automated | No | Yes (automation potential scoring) |
| What causes friction | No | Yes (friction point identification) |
| Which skills compose into workflows | No | Yes (chain discovery + archetypes) |

### Top 3 skills to build
1. **Docs and architecture consistency audit** — high frequency, fully automatable, protects against drift
2. **Planning-to-implementation handoff** — reduces rework on the most common upstream activity
3. **Command surface and help generation** — immediate CLI productivity gains from generated scaffolds
